In [1]:
import pandas as pd 
import numpy as np 

from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression

from scipy.stats import kendalltau, linregress
from scipy.stats import weightedtau
from utils import normalize_columns, encode_categorical, centralize, bootstrap

## Preparing input data 

In [2]:
print('Preparing data...')
INPUT_FILE = '../inputs/transf_scores.csv'
# Read the input data
df_full = pd.read_csv(INPUT_FILE)

# ... filters the data by datasets and scorers
ALL_SCORERS = df_full['transf_metric'].unique().tolist()
ALL_DBS =  df_full['dataset'].unique().tolist()
SIGNIFICANCE_LEVEL = 0.05

Preparing data...


### Define variables

In [3]:
DATASET_ORDER = ['caltech101',
                'sun397',
                'voc2007',
                'flowers102',
                'oxfordpets',
                'aircraft',
                'dtd',
                'stanfordcars',
                'brain_tumor_kaggle',
                'breakhis',
                'skin_splits']

SCORER_ORDER = ["ncti_score",        
                "etran_energy_score",
                "pactran_score",     
                "gbc_score",         
                "parc_score",        
                "nleep_score",       
                "leep_score",        
                "logme_score",       
                "tmi_score",         
                "nce_score",         
                "sfda_score",        
                "hscore_score",      
                "reg_hscore_score",  ]



LINEAR_ABLATION_NAMES = {'result_overall': 'Pool. completo',
                         'result_pool_per_scorer': 'Pool. per scorer',
                         'result_pool_per_scorer_dataset': 'Pool. per scorer-dataset',}

DECIMALS = 3

BTB_SCHEME = ['bh3_I+BEST']
# BTB_SCHEME = ['bh3_BEST']
SCORERS_TO_KEEP = [
                    'imagenet',
                   'pactran_score',
                   'etran_energy_score',
                   'ncti_score' 
                  ]


## Define functions to compute bootstrapping and regression 

In [4]:
def is_constant_array(arr):
        return np.all(arr == arr[0])

def kendall_taus(x, y):
    if is_constant_array(x):
        if is_constant_array(y):
            # This should be an exceedingly rare situation in which we sampled the same value n times
            pass
        else:
            # If the scores are constant when the metric is not, we assign no correlation/predictive power to the sample
            # we handle this explicitly because kendalltau/weightedtau returns NaN in this case
            return 0, 0
    tau, _ = kendalltau(x, y)
    if np.isnan(tau) or np.isinf(tau):
        print('Warning: Nonfinite tau', tau, 'for', x, y)
    wtau, _ = weightedtau(x, y)
    return tau, wtau

## Linear models ablations 

In [5]:
def linear_pooling_overall(df_train, df_test):
    # Linear pooling considering ALL scorers and datasets  
    score = df_train['z_transf_score']
    metric = df_train['z_test_score']
    model = linregress(score, metric)
    df_test = df_test.head(10)
    # inference
    preds = model.slope * df_test['z_transf_score'] + model.intercept 
    # calculate correlation between predictions and true scorers
    kendall_all_pooling = bootstrap((preds.to_list(), 
                                    df_test['z_test_score'].to_list()), kendall_taus, n_bootstraps=1000)

    return 100*np.array([k[0] for k in kendall_all_pooling]).mean().round(decimals=DECIMALS), preds.to_numpy()

def linear_pooling_per_scorer(df_train, df_test):
    # # Linear pooling per scorer 
    models_dict = dict()
    for scorer_name in df_train['transf_metric'].unique():
        df_scorer_train = df_train[df_train['transf_metric'] == scorer_name]
        score = df_scorer_train['z_transf_score']
        metric = df_scorer_train['z_test_score']
        models_dict[scorer_name] = linregress(score, metric)
    
    # inference -> aggregate by scorer 
    aggregated_predictions = []
    for scorer_name in df_test['transf_metric'].unique():
        df_scorer_test = df_test[df_test['transf_metric'] == scorer_name]
        model = models_dict[scorer_name]
        y_inf = model.slope * df_scorer_test['z_transf_score'] + model.intercept 
        aggregated_predictions.append(np.array(y_inf).reshape(1, -1))
    
    # aggregate predictions 
    aggregated_predictions = np.concatenate(aggregated_predictions).mean(axis=0)
    transfer_performances = df_test.head(10)['z_test_score'].to_list()
    kendall_per_scorer_pooling = bootstrap((aggregated_predictions, transfer_performances), 
                                           kendall_taus, 
                                           n_bootstraps=1000)
    kendall_per_scorer_pooling = np.array([k[0] for k in kendall_per_scorer_pooling])
    
    return 100*kendall_per_scorer_pooling.mean().round(decimals=DECIMALS), aggregated_predictions

def linear_pooling_per_score_dataset(df_train, df_test):
    models_dict = dict()
    # train one regression per scorer and dataset
    for dataset_name in df_train['dataset'].unique():
        models_dict[dataset_name] = dict()
        for scorer_name in df_train['transf_metric'].unique():
            df_scorer_train = df_train[(df_train['transf_metric'] == scorer_name) & 
                                       (df_train['dataset'] == dataset_name)]
            score = df_scorer_train['z_transf_score']
            metric = df_scorer_train['z_test_score']
            models_dict[dataset_name][scorer_name] = linregress(score, metric)
    
    # inference 
    aggregated_predictions = []
    # sweep over all datasets in traning and all scorers in test
    for dataset_name in df_train['dataset'].unique():
        for scorer_name in df_test['transf_metric'].unique():
            df_scorer_test = df_test[(df_test['transf_metric'] == scorer_name)] # there will be only 10 measurements             
            model = models_dict[dataset_name][scorer_name]
            preds = model.slope * df_scorer_test['z_transf_score'] + model.intercept
            aggregated_predictions.append(np.array(preds).reshape(1, -1)) # 1 x 10 
    
    # aggregate predictions 
    aggregated_predictions = np.concatenate(aggregated_predictions).mean(axis=0) 
    transfer_performances = df_test.head(10)['z_test_score'].to_list()
    kendall_per_scorer_pooling = bootstrap((aggregated_predictions, transfer_performances), 
                                            kendall_taus, 
                                            n_bootstraps=1000)
    kendall_per_scorer_pooling = np.array([k[0] for k in kendall_per_scorer_pooling])
    return 100*kendall_per_scorer_pooling.mean().round(decimals=DECIMALS), aggregated_predictions

### SKLearn Classifier

In [6]:
def sklearn_regression(df_train, df_test, regression_fn):
    # prepare data for input 
    train_data = dict()
    train_data['z_transf_score'] = list()
    train_data['z_test_score'] = list()
    for idx, scorer in enumerate(SCORERS_TO_KEEP):
        df_scorer = df_train[df_train['transf_metric'] == scorer]
        train_data['z_transf_score'].append(df_scorer['z_transf_score'].to_list())
        
        # Enter here only once to avoid repeated samples
        if idx == 0:
            train_data['z_test_score'].extend(df_scorer['z_test_score'].to_list())

    # concatenate scorers information as features (columns)
    train_data['z_transf_score'] = np.column_stack(train_data['z_transf_score']) # Shape: (N_Archs*N_TrainDatasets, N_Trainscorers)  
    train_data['z_test_score'] = np.array(train_data['z_test_score']) # Shape: (N_Archs*N_TrainDatasets, 1)
    # print(f"{train_data['z_transf_score'].shape=} / {train_data['z_test_score'].shape=}")

    test_data = dict()
    test_data['z_transf_score'] = list()
    test_data['z_test_score'] = list()
    for idx, scorer in enumerate(SCORERS_TO_KEEP):
        df_scorer = df_test[df_test['transf_metric'] == scorer]
        test_data['z_transf_score'].append(df_scorer['z_transf_score'].to_list())
        # Enter here only once to avoid repeated samples
        if idx == 0:
            test_data['z_test_score'].extend(df_scorer['z_test_score'].to_list())

    # concatenate scorers information as features (columns)
    test_data['z_transf_score'] = np.column_stack(test_data['z_transf_score']) # Shape: (N_Archs*N_TestDatasets, N_Trainscorers)  
    test_data['z_test_score'] = np.array(test_data['z_test_score']) # Shape: (N_Archs*N_TestDatasets, 1)

    # print(f"{test_data['z_transf_score'].shape=} / {test_data['z_test_score'].shape=}")
    # train and test SVM for regression 
    regression_fn.fit(X=train_data['z_transf_score'], 
            y=train_data['z_test_score'])
    predictions = regression_fn.predict(test_data['z_transf_score']) # Shape (N_Archs, 1) 
    # print(f"{predictions.shape=}")
    kendall_regression = bootstrap((predictions, test_data['z_test_score']), 
                            kendall_taus, 
                            n_bootstraps=1000)
    
    kendall_regression = np.array([k[0] for k in kendall_regression])
    return 100*kendall_regression.mean().round(decimals=DECIMALS), predictions#.round(decimals=DECIMALS)


### Same scheme as in BtB + 3 top 

In [7]:
# Iterate over all available DBS to vary the test DB (leave-one-dataset-out evaluation scheme)
summary = []
for TEST_DB in ALL_DBS:
    TRAIN_DBS = [db for db in ALL_DBS if db != TEST_DB]

    SELECTED_SCORERS = SCORERS_TO_KEEP.copy()
    DATASETS = sorted(set(TRAIN_DBS) - {TEST_DB}) + [TEST_DB]
    df = df_full[df_full['transf_metric'].isin(SELECTED_SCORERS) & df_full['dataset'].isin(DATASETS)].copy()
    group_by = ['transf_metric', 'dataset']
    columns_to_normalize = ['test_score', 'transf_score']
    normalize_columns(df, columns=columns_to_normalize, group_by=group_by) # in-place normalization 
    
    # Encode the categorical variables as integers
    translation = encode_categorical(df, 
                                     columns=['model', 'transf_metric', 'dataset'], 
                                     encoding = dict( model=None, transf_metric=None, dataset=dict(enumerate(DATASETS, start=1)) ))
    
    # Selects and validates the data
    df_train = df[df['dataset'].isin(TRAIN_DBS)]
    df_test = df[df['dataset'] == TEST_DB]
    if len(df_train) == 0:
        raise ValueError('No training data meets the criteria!')
    if len(df_test) == 0:
        raise ValueError('No test data meets the criteria!')

    df_target_check = df_test.groupby('model')['test_score'].transform(centralize).abs().max()
    if df_target_check > 1e-6:
        raise ValueError('The test scores are not constant for each model on the test dataset!')

    if df_test['classes'].nunique() != 1:
        raise ValueError('The test dataset has different class counts on different entries!')


    print(f'[{TEST_DB=}] Fitting models...')
    results = {}
    outputs = {}
    
    result_overall, preds_overall = linear_pooling_overall(df_train, df_test)
    
    result_pool_per_scorer, preds_per_scorer = linear_pooling_per_scorer(df_train, df_test)

    result_pool_per_scorer_dataset, preds_per_scorer_dataset = linear_pooling_per_score_dataset(df_train, df_test)

    results_svm, preds_svm = sklearn_regression(df_train, df_test, regression_fn=SVR())

    results_least_squares, preds_least_squares = sklearn_regression(df_train, df_test, regression_fn=LinearRegression())

    print(f"{preds_overall.shape=} / {preds_per_scorer.shape=} / {preds_per_scorer_dataset.shape=}")
    
    df_test = df_test.head(10).reset_index()

    for row_id, row in df_test.iterrows():
        summary.append([TEST_DB, row['model'], row['test_score'], row['classes'], 'linear_pooling_overall', preds_overall[row_id]])
    
    for row_id, row in df_test.iterrows():
        summary.append([TEST_DB, row['model'], row['test_score'], row['classes'], 'linear_pooling_per_scorer', preds_per_scorer[row_id]])
    
    for row_id, row in df_test.iterrows():
        summary.append([TEST_DB, row['model'], row['test_score'], row['classes'], 'linear_pooling_per_score_dataset', preds_per_scorer_dataset[row_id]])

    for row_id, row in df_test.iterrows():
        summary.append([TEST_DB, row['model'], row['test_score'], row['classes'], 'svm', preds_svm[row_id]])
    
    for row_id, row in df_test.iterrows():
        summary.append([TEST_DB, row['model'], row['test_score'], row['classes'], 'least_squares', preds_least_squares[row_id]])

    
    
df_summary_top_scheme = pd.DataFrame(summary, columns=['dataset', 
                                                       'model', 
                                                       'test_score', 
                                                       'classes', 
                                                       'transf_metric',
                                                       'transf_score'])
display(df_summary_top_scheme)

[TEST_DB='sun397'] Fitting models...
preds_overall.shape=(10,) / preds_per_scorer.shape=(10,) / preds_per_scorer_dataset.shape=(10,)
[TEST_DB='aircraft'] Fitting models...
preds_overall.shape=(10,) / preds_per_scorer.shape=(10,) / preds_per_scorer_dataset.shape=(10,)
[TEST_DB='caltech101'] Fitting models...
preds_overall.shape=(10,) / preds_per_scorer.shape=(10,) / preds_per_scorer_dataset.shape=(10,)
[TEST_DB='oxfordpets'] Fitting models...
preds_overall.shape=(10,) / preds_per_scorer.shape=(10,) / preds_per_scorer_dataset.shape=(10,)
[TEST_DB='flowers102'] Fitting models...
preds_overall.shape=(10,) / preds_per_scorer.shape=(10,) / preds_per_scorer_dataset.shape=(10,)
[TEST_DB='dtd'] Fitting models...
preds_overall.shape=(10,) / preds_per_scorer.shape=(10,) / preds_per_scorer_dataset.shape=(10,)
[TEST_DB='stanfordcars'] Fitting models...
preds_overall.shape=(10,) / preds_per_scorer.shape=(10,) / preds_per_scorer_dataset.shape=(10,)
[TEST_DB='voc2007'] Fitting models...
preds_overall.

,dataset,model,test_score,classes,transf_metric,transf_score
0,sun397,densenet121,0.497280,397,linear_pooling_overall,0.051819
1,sun397,densenet161,0.514458,397,linear_pooling_overall,0.611841
2,sun397,densenet169,0.514710,397,linear_pooling_overall,0.293308
3,sun397,efficientnet_b0,0.522519,397,linear_pooling_overall,0.726580
4,sun397,mobilenetv2_050,0.411033,397,linear_pooling_overall,-1.705294
...,...,...,...,...,...,...
545,skin_splits,mobilenetv2_100,0.862750,2,least_squares,-0.142551
546,skin_splits,resnet18,0.877213,2,least_squares,-0.440803
547,skin_splits,resnet34,0.871371,2,least_squares,-0.540125
548,skin_splits,resnet50,0.837430,2,least_squares,-0.082571


In [8]:
df_summary_top_scheme.head(10)

,dataset,model,test_score,classes,transf_metric,transf_score
0,sun397,densenet121,0.497280,397,linear_pooling_overall,0.051819
1,sun397,densenet161,0.514458,397,linear_pooling_overall,0.611841
2,sun397,densenet169,0.514710,397,linear_pooling_overall,0.293308
3,sun397,efficientnet_b0,0.522519,397,linear_pooling_overall,0.726580
4,sun397,mobilenetv2_050,0.411033,397,linear_pooling_overall,-1.705294
5,sun397,mobilenetv2_100,0.500403,397,linear_pooling_overall,-0.250146
6,sun397,resnet18,0.475013,397,linear_pooling_overall,-0.916623
7,sun397,resnet34,0.484736,397,linear_pooling_overall,-0.180143
8,sun397,resnet50,0.562217,397,linear_pooling_overall,0.403076
9,sun397,vit_small_patch16_224,0.666096,397,linear_pooling_overall,0.965584


In [9]:
df_summary_top_scheme.tail(10)

,dataset,model,test_score,classes,transf_metric,transf_score
540,skin_splits,densenet121,0.869602,2,least_squares,-0.242184
541,skin_splits,densenet161,0.888792,2,least_squares,0.878941
542,skin_splits,densenet169,0.882190,2,least_squares,0.328462
543,skin_splits,efficientnet_b0,0.879774,2,least_squares,0.326039
544,skin_splits,mobilenetv2_050,0.766614,2,least_squares,-1.533207
545,skin_splits,mobilenetv2_100,0.862750,2,least_squares,-0.142551
546,skin_splits,resnet18,0.877213,2,least_squares,-0.440803
547,skin_splits,resnet34,0.871371,2,least_squares,-0.540125
548,skin_splits,resnet50,0.837430,2,least_squares,-0.082571
549,skin_splits,vit_small_patch16_224,0.887396,2,least_squares,1.448001


In [ ]:
# # Concatenate with BtB results 
# # bh3_MID 
# btb_path = '/experimentos/codes/doutorado/btb/gradstat/inputs/transf_scores_with_b2b.csv'
# btb_df = pd.read_csv(btb_path)

# btb_subset_df = btb_df[btb_df['transf_metric'].isin(BTB_SCHEME)]

In [ ]:
# btb_subset_df

,dataset,model,test_score,classes,transf_metric,transf_score
2310,aircraft,densenet121,0.471373,100,bh3_I+BEST,0.361431
2311,aircraft,densenet161,0.563975,100,bh3_I+BEST,0.794499
2312,aircraft,densenet169,0.525169,100,bh3_I+BEST,0.499326
2313,aircraft,efficientnet_b0,0.529768,100,bh3_I+BEST,0.142606
2314,aircraft,mobilenetv2_050,0.275437,100,bh3_I+BEST,-1.955653
...,...,...,...,...,...,...
2415,voc2007,mobilenetv2_100,0.725428,20,bh3_I+BEST,-0.164928
2416,voc2007,resnet18,0.710734,20,bh3_I+BEST,-0.431373
2417,voc2007,resnet34,0.731867,20,bh3_I+BEST,-0.002097
2418,voc2007,resnet50,0.779262,20,bh3_I+BEST,0.365430


In [ ]:
# pd.concat([df_summary_top_scheme, btb_subset_df]).to_csv('../inputs/btb_plus_lin_ablations.csv', index=False)